#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.window import Window
from pyspark.sql.functions import trim, col, length

#Read Bronze Table

In [0]:
df = spark.table("demo_project.bronze.erp_cust_az12")

##Sanity checks

In [0]:
df.limit(10).display()

#Silver Transformations

##Renaming Columns


In [0]:
rename_map = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "gender"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity Checks 

In [0]:
df.limit(10).display()

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Customer ID Cleanup

In [0]:
df = df.withColumn(
    "customer_id",
    F.when(col("customer_id").startswith("NAS"),
           F.substring(col("customer_id"), 4, F.length(col("customer_id"))))
           .otherwise(col("customer_id"))
)

##Sanity checks

In [0]:
df.limit(10).display()

##Birthdate validation

In [0]:
df = df.withColumn(
    "birth_date",
    F.when(col("birth_date") > F.current_date(), None)
    .otherwise(col("birth_date"))
)

##Gender Normalization

In [0]:
df = df.withColumn(
    "gender",
    F.when(F.upper(col("gender")).isin("F", "FEMALE"), "Female")
    .when(F.upper(col("gender")).isin("M", "MALE"), "Male")
    .otherwise("N/A")
)

##Sanity checks

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("demo_project.silver.erp_customers")

##Sanity checks

In [0]:
%sql
select * from demo_project.silver.erp_customers limit 10;